## Phase-locking value

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from hypyp import analyses
%pip install mne-qt-browser
%pip install PySide6
mne.viz.set_browser_backend('qt')

/Users/felix/Documents/MPI/EEG_hyperscanning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Using qt as 2D backend.


Sanity check

In [3]:
epochs_R = mne.read_epochs("data/ds007471/derivatives/sub-01_R-epo.fif", preload=True)
epochs_L = mne.read_epochs("data/ds007471/derivatives/sub-01_L-epo.fif", preload=True)

print(epochs_R)
print(epochs_L)

print("Number of trials:", len(epochs_R))
print("R channels:", len(epochs_R.ch_names))
print("L channels:", len(epochs_L.ch_names))
print("Sampling frequency:", epochs_R.info["sfreq"])
assert len(epochs_R) == len(epochs_L)
assert epochs_R.info["sfreq"] == epochs_L.info["sfreq"]

Reading /Users/felix/Documents/MPI/EEG_hyperscanning/data/ds007471/derivatives/sub-01_R-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    5000.00 ms
        0 CTF compensation matrices available
Not setting metadata
57 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/felix/Documents/MPI/EEG_hyperscanning/data/ds007471/derivatives/sub-01_L-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    5000.00 ms
        0 CTF compensation matrices available
Not setting metadata
57 matching events found
No baseline correction applied
0 projection items activated
<EpochsFIF | 57 events (all good), -0.5 – 5 s (baseline -0.5 – 0 s), ~64.6 MiB, data loaded,
 'trial_start': 57>
<EpochsFIF | 57 events (all good), -0.5 – 5 s (baseline -0.5 – 0 s), ~64.6 MiB, data loaded,
 'trial_start': 57>
Number of trials: 57
R channels: 27
L channels: 27
Sampling frequency: 1000.0


Get EEG data as NumPy arrays

In [4]:
data_R = epochs_R.get_data()
data_L = epochs_L.get_data()

print(data_R.shape)
print(data_L.shape)

(57, 27, 5501)
(57, 27, 5501)


Apply band-pass to filter for Theta band

In [5]:
epochs_R_theta = epochs_R.copy().filter(
    l_freq=4,
    h_freq=8,
    picks="eeg"
)

epochs_L_theta = epochs_L.copy().filter(
    l_freq=4,
    h_freq=8,
    picks="eeg"
)

Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter le